In [1]:
"""
╔══════════════════════════════════════════════════════════════╗
║              EvoForest 2.0 — Fraud Detection                 ║
║  Features: Hall of Fame | Crossover | Directed Evolution     ║
║            Internal Validation | Early Stopping              ║
╚══════════════════════════════════════════════════════════════╝

Dataset: Credit Card Fraud Detection (Kaggle)
https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
"""

import numpy as np
import pandas as pd
import time
import random
from copy import deepcopy
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, recall_score, precision_score,
                             classification_report, confusion_matrix)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')


# ══════════════════════════════════════════════════════════════
#  SECTION 1: CONFIGURATION
# ══════════════════════════════════════════════════════════════

CONFIG = {
    # Dataset
    "data_path": "/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv",          # ← ضع مسار الملف هنا

    # Population
    "population_size": 20,                  # عدد الأشجار في كل جيل
    "n_generations": 15,                    # عدد الأجيال

    # Evolution
    "mutation_rate": 0.3,                   # احتمال الطفرة
    "elite_size": 4,                        # عدد الأشجار النخبة (تبقى بلا تغيير)

    # Hall of Fame ← الجديد
    "hall_of_fame_size": 3,                 # عدد أفضل الأشجار المحفوظة

    # Early Stopping ← الجديد
    "patience": 3,                          # أجيال بلا تحسن قبل التوقف

    # Directed Evolution ← الجديد
    "directed_evolution": True,             # تفعيل التطور الموجه

    # Data Split
    "test_size": 0.2,
    "val_size": 0.15,                       # Internal Validation ← الجديد

    # Scoring metric
    "metric": "f1",                         # f1 | recall | precision

    # Reproducibility
    "random_seed": 42,
}

# Hyperparameter search space
PARAM_SPACE = {
    "max_depth":            [3, 4, 5, 6, 7, 8, 10, 12, 15],
    "min_samples_split":    [2, 5, 10, 20, 50],
    "min_samples_leaf":     [1, 2, 5, 10, 20],
    "max_features":         [0.3, 0.5, 0.7, 0.9, "sqrt", "log2"],
    "criterion":            ["gini", "entropy"],
    "class_weight":         ["balanced", None, {0: 1, 1: 5}, {0: 1, 1: 10}],
}


# ══════════════════════════════════════════════════════════════
#  SECTION 2: DATA LOADING & PREPROCESSING
# ══════════════════════════════════════════════════════════════

def load_and_prepare_data(config):
    """تحميل البيانات وتقسيمها ثلاثياً: Train / Validation / Test"""
    print("=" * 60)
    print("  📂 Loading Data...")
    print("=" * 60)

    df = pd.read_csv(config["data_path"])
    print(f"  ✅ Shape: {df.shape}")
    print(f"  ✅ Fraud cases: {df['Class'].sum()} ({df['Class'].mean()*100:.3f}%)")

    X = df.drop("Class", axis=1)
    y = df["Class"]

    # Scale Amount and Time
    scaler = StandardScaler()
    X = X.copy()
    X["Amount"] = scaler.fit_transform(X[["Amount"]])
    X["Time"]   = scaler.fit_transform(X[["Time"]])

    # Split: Train+Val / Test
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y,
        test_size=config["test_size"],
        random_state=config["random_seed"],
        stratify=y
    )

    # Split: Train / Validation
    val_ratio = config["val_size"] / (1 - config["test_size"])
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=val_ratio,
        random_state=config["random_seed"],
        stratify=y_temp
    )

    print(f"\n  📊 Data Split (Internal Validation):")
    print(f"     Train      : {X_train.shape[0]:,} samples")
    print(f"     Validation : {X_val.shape[0]:,} samples")
    print(f"     Test       : {X_test.shape[0]:,} samples")
    print()

    return X_train, X_val, X_test, y_train, y_val, y_test


# ══════════════════════════════════════════════════════════════
#  SECTION 3: TREE (INDIVIDUAL) OPERATIONS
# ══════════════════════════════════════════════════════════════

def random_params():
    """إنشاء معاملات عشوائية لشجرة جديدة"""
    return {k: random.choice(v) for k, v in PARAM_SPACE.items()}


def build_tree(params, seed=None):
    """بناء DecisionTree من معاملات"""
    return DecisionTreeClassifier(
        max_depth=params["max_depth"],
        min_samples_split=params["min_samples_split"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
        criterion=params["criterion"],
        class_weight=params["class_weight"],
        random_state=seed if seed else random.randint(0, 9999)
    )


def evaluate_tree(tree, X, y, metric="f1"):
    """تقييم شجرة على مجموعة بيانات"""
    preds = tree.predict(X)
    if metric == "f1":
        return f1_score(y, preds, zero_division=0)
    elif metric == "recall":
        return recall_score(y, preds, zero_division=0)
    elif metric == "precision":
        return precision_score(y, preds, zero_division=0)


# ══════════════════════════════════════════════════════════════
#  SECTION 4: HALL OF FAME ← الجديد
# ══════════════════════════════════════════════════════════════

class HallOfFame:
    """
    خزنة تحفظ أفضل الأشجار عبر كل الأجيال.
    - لا تُحذف أي شجرة محفوظة مهما تراجعت الأجيال القادمة
    - النموذج النهائي = بطل الأبطال من الخزنة
    """

    def __init__(self, size=3):
        self.size = size
        self.champions = []       # list of (score, params, tree)

    def update(self, candidates):
        """
        candidates: list of (score, params, trained_tree)
        """
        for score, params, tree in candidates:
            self.champions.append((score, deepcopy(params), deepcopy(tree)))

        # الترتيب والاحتفاظ بالأفضل فقط
        self.champions.sort(key=lambda x: x[0], reverse=True)
        self.champions = self.champions[:self.size]

    def best_score(self):
        if self.champions:
            return self.champions[0][0]
        return 0.0

    def best_tree(self):
        if self.champions:
            return self.champions[0][2]
        return None

    def display(self):
        print("\n  🏆 Hall of Fame:")
        for i, (score, params, _) in enumerate(self.champions):
            print(f"     #{i+1}  Score={score:.4f} | "
                  f"depth={params['max_depth']} | "
                  f"features={params['max_features']} | "
                  f"criterion={params['criterion']}")


# ══════════════════════════════════════════════════════════════
#  SECTION 5: CROSSOVER ← الجديد
# ══════════════════════════════════════════════════════════════

def crossover(parent1_params, parent2_params):
    """
    تزاوج بين شجرتين:
    - max_depth, criterion, class_weight  ← من الأب الأول
    - max_features, min_samples_*         ← من الأب الثاني
    """
    child_params = {
        "max_depth":         parent1_params["max_depth"],
        "criterion":         parent1_params["criterion"],
        "class_weight":      parent1_params["class_weight"],
        "max_features":      parent2_params["max_features"],
        "min_samples_split": parent2_params["min_samples_split"],
        "min_samples_leaf":  parent2_params["min_samples_leaf"],
    }
    return child_params


# ══════════════════════════════════════════════════════════════
#  SECTION 6: DIRECTED EVOLUTION ← الجديد
# ══════════════════════════════════════════════════════════════

class DirectedEvolution:
    """
    بدلاً من الطفرة العشوائية الكاملة:
    - تراقب المعاملات التي تكررت في الأشجار الفائزة
    - تجعل الطفرات الجديدة تحوم حولها بنسبة أعلى
    """

    def __init__(self):
        self.winner_history = {k: [] for k in PARAM_SPACE}

    def record_winners(self, top_params_list):
        """تسجيل معاملات الأشجار الفائزة"""
        for params in top_params_list:
            for k, v in params.items():
                self.winner_history[k].append(v)

    def smart_mutate(self, params, mutation_rate=0.3):
        """
        طفرة ذكية:
        - إذا وُجد تاريخ فائزين → اختر من بينهم (70%) أو عشوائي (30%)
        - إذا لا يوجد تاريخ   → عشوائي كامل
        """
        new_params = deepcopy(params)
        for key in PARAM_SPACE:
            if random.random() < mutation_rate:
                history = self.winner_history[key]
                if history and random.random() < 0.7:
                    # طفرة موجهة نحو الفائزين
                    new_params[key] = random.choice(history)
                else:
                    # طفرة عشوائية
                    new_params[key] = random.choice(PARAM_SPACE[key])
        return new_params


# ══════════════════════════════════════════════════════════════
#  SECTION 7: EVOLUTION ENGINE
# ══════════════════════════════════════════════════════════════

def initialize_population(size):
    """إنشاء جيل ابتدائي عشوائي"""
    return [random_params() for _ in range(size)]


def selection(scored_population, elite_size):
    """
    اختيار النخبة:
    - أفضل (elite_size) شجرة تبقى مباشرة
    - الباقي يُستخدم كآباء للتزاوج والطفرة
    """
    scored_population.sort(key=lambda x: x[0], reverse=True)
    elites  = [p for _, p, _ in scored_population[:elite_size]]
    parents = [p for _, p, _ in scored_population]
    return elites, parents


def create_next_generation(elites, parents, population_size,
                           directed_evo, mutation_rate, elite_size):
    """بناء الجيل التالي: نخبة + أبناء تزاوج + طفرات"""
    next_gen = list(elites)                # النخبة تنتقل مباشرة

    while len(next_gen) < population_size:
        # اختيار عشوائي للآباء (الأفضل لديهم وزن أكبر)
        weights = [1 / (i + 1) for i in range(len(parents))]
        p1, p2 = random.choices(parents, weights=weights, k=2)

        # Crossover
        child = crossover(p1, p2)

        # Directed Mutation
        child = directed_evo.smart_mutate(child, mutation_rate)
        next_gen.append(child)

    return next_gen[:population_size]


def run_evolution(X_train, X_val, X_test, y_train, y_val, y_test, config):
    """المحرك التطوري الرئيسي"""

    random.seed(config["random_seed"])
    np.random.seed(config["random_seed"])

    metric        = config["metric"]
    pop_size      = config["population_size"]
    n_generations = config["n_generations"]
    elite_size    = config["elite_size"]
    mutation_rate = config["mutation_rate"]
    patience      = config["patience"]

    # الأدوات الجديدة
    hall_of_fame   = HallOfFame(size=config["hall_of_fame_size"])
    directed_evo   = DirectedEvolution()

    population     = initialize_population(pop_size)
    best_val_score = 0.0
    no_improve     = 0
    history        = []

    start_time = time.time()

    print("\n" + "=" * 60)
    print("  🌿 EvoForest 2.0 — Evolution Started")
    print("=" * 60)

    for gen in range(1, n_generations + 1):

        # ── تدريب وتقييم كل الأشجار ──────────────────────────
        scored = []
        for params in population:
            tree = build_tree(params)
            tree.fit(X_train, y_train)
            # التقييم على Validation (Internal Validation)
            val_score = evaluate_tree(tree, X_val, y_val, metric)
            scored.append((val_score, params, tree))

        scored.sort(key=lambda x: x[0], reverse=True)

        gen_best_score  = scored[0][0]
        gen_best_params = scored[0][1]
        gen_best_tree   = scored[0][2]

        # ── Hall of Fame Update ───────────────────────────────
        hall_of_fame.update([(s, p, t) for s, p, t in scored[:config["hall_of_fame_size"]]])

        # ── Directed Evolution: تسجيل الفائزين ───────────────
        top_params = [p for _, p, _ in scored[:elite_size]]
        directed_evo.record_winners(top_params)

        # ── Early Stopping ────────────────────────────────────
        if gen_best_score > best_val_score + 1e-5:
            best_val_score = gen_best_score
            no_improve = 0
        else:
            no_improve += 1

        history.append({
            "generation": gen,
            "best_val_score": gen_best_score,
            "hall_of_fame_best": hall_of_fame.best_score(),
            "no_improve": no_improve,
        })

        elapsed = time.time() - start_time
        print(f"  Gen {gen:02d}/{n_generations} | "
              f"Val {metric.upper()}={gen_best_score:.4f} | "
              f"HoF={hall_of_fame.best_score():.4f} | "
              f"NoImprove={no_improve}/{patience} | "
              f"⏱ {elapsed:.1f}s")

        if no_improve >= patience:
            print(f"\n  ⏹ Early Stopping at generation {gen} "
                  f"(no improvement for {patience} generations)")
            break

        # ── الجيل التالي ──────────────────────────────────────
        elites, parents = selection(scored, elite_size)
        population = create_next_generation(
            elites, parents, pop_size,
            directed_evo, mutation_rate, elite_size
        )

    total_time = time.time() - start_time

    # ── الاختبار النهائي على Test Set ────────────────────────
    champion = hall_of_fame.best_tree()

    print("\n" + "=" * 60)
    print("  🏅 FINAL EVALUATION ON TEST SET")
    print("=" * 60)

    y_pred = champion.predict(X_test)

    f1  = f1_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    pre = precision_score(y_test, y_pred, zero_division=0)

    print(f"\n  F1-Score  : {f1:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  Precision : {pre:.4f}")
    print(f"  Train Time: {total_time:.1f} seconds")

    print("\n  📋 Full Classification Report:")
    print(classification_report(y_test, y_pred,
                                target_names=["Normal", "Fraud"]))

    print("\n  📊 Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(f"     TN={cm[0][0]:,}  FP={cm[0][1]:,}")
    print(f"     FN={cm[1][0]:,}  TP={cm[1][1]:,}")

    hall_of_fame.display()

    print("\n" + "=" * 60)
    print(f"  ✅ EvoForest 2.0 Complete — {total_time:.1f}s")
    print("=" * 60)

    return {
        "model": champion,
        "hall_of_fame": hall_of_fame,
        "history": history,
        "metrics": {"f1": f1, "recall": rec, "precision": pre},
        "training_time": total_time,
    }


# ══════════════════════════════════════════════════════════════
#  SECTION 8: COMPARISON WITH RANDOM FOREST
# ══════════════════════════════════════════════════════════════

def compare_with_random_forest(X_train, X_val, X_test,
                                y_train, y_val, y_test, seed=42):
    """مقارنة EvoForest مع Random Forest الكلاسيكي"""
    from sklearn.ensemble import RandomForestClassifier

    print("\n" + "=" * 60)
    print("  🌲 Random Forest Baseline")
    print("=" * 60)

    # دمج Train+Val للتدريب (نفس الطريقة)
    X_tr = pd.concat([X_train, X_val])
    y_tr = pd.concat([y_train, y_val])

    start = time.time()
    rf = RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=seed
    )
    rf.fit(X_tr, y_tr)
    rf_time = time.time() - start

    y_pred = rf.predict(X_test)

    f1  = f1_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    pre = precision_score(y_test, y_pred, zero_division=0)

    print(f"\n  F1-Score  : {f1:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  Precision : {pre:.4f}")
    print(f"  Train Time: {rf_time:.1f} seconds")

    return {"f1": f1, "recall": rec, "precision": pre, "time": rf_time}


def print_comparison_table(evo_results, rf_results):
    """طباعة جدول المقارنة النهائي"""
    em = evo_results["metrics"]
    et = evo_results["training_time"]

    print("\n" + "=" * 60)
    print("  📊 FINAL COMPARISON TABLE")
    print("=" * 60)
    print(f"  {'Metric':<20} {'EvoForest 2.0':>15} {'Random Forest':>15}")
    print(f"  {'-'*50}")
    print(f"  {'F1 (Fraud)':<20} {em['f1']:>15.4f} {rf_results['f1']:>15.4f}")
    print(f"  {'Recall':<20} {em['recall']:>15.4f} {rf_results['recall']:>15.4f}")
    print(f"  {'Precision':<20} {em['precision']:>15.4f} {rf_results['precision']:>15.4f}")
    print(f"  {'Training Time (s)':<20} {et:>15.1f} {rf_results['time']:>15.1f}")
    print("=" * 60)


# ══════════════════════════════════════════════════════════════
#  SECTION 9: MAIN
# ══════════════════════════════════════════════════════════════

if __name__ == "__main__":

    # 1. تحميل البيانات
    X_train, X_val, X_test, y_train, y_val, y_test = load_and_prepare_data(CONFIG)

    # 2. تشغيل EvoForest 2.0
    evo_results = run_evolution(
        X_train, X_val, X_test,
        y_train, y_val, y_test,
        CONFIG
    )

    # 3. مقارنة مع Random Forest
    rf_results = compare_with_random_forest(
        X_train, X_val, X_test,
        y_train, y_val, y_test,
        seed=CONFIG["random_seed"]
    )

    # 4. جدول المقارنة النهائي
    print_comparison_table(evo_results, rf_results)

  📂 Loading Data...
  ✅ Shape: (284807, 31)
  ✅ Fraud cases: 492 (0.173%)

  📊 Data Split (Internal Validation):
     Train      : 185,124 samples
     Validation : 42,721 samples
     Test       : 56,962 samples


  🌿 EvoForest 2.0 — Evolution Started
  Gen 01/15 | Val F1=0.8296 | HoF=0.8296 | NoImprove=0/3 | ⏱ 80.9s
  Gen 02/15 | Val F1=0.8369 | HoF=0.8369 | NoImprove=0/3 | ⏱ 174.5s
  Gen 03/15 | Val F1=0.8120 | HoF=0.8369 | NoImprove=1/3 | ⏱ 270.7s
  Gen 04/15 | Val F1=0.8571 | HoF=0.8571 | NoImprove=0/3 | ⏱ 377.2s
  Gen 05/15 | Val F1=0.8593 | HoF=0.8593 | NoImprove=0/3 | ⏱ 493.3s
  Gen 06/15 | Val F1=0.8382 | HoF=0.8593 | NoImprove=1/3 | ⏱ 602.2s
  Gen 07/15 | Val F1=0.8227 | HoF=0.8593 | NoImprove=2/3 | ⏱ 719.9s
  Gen 08/15 | Val F1=0.8358 | HoF=0.8593 | NoImprove=3/3 | ⏱ 844.8s

  ⏹ Early Stopping at generation 8 (no improvement for 3 generations)

  🏅 FINAL EVALUATION ON TEST SET

  F1-Score  : 0.8556
  Recall    : 0.7857
  Precision : 0.9390
  Train Time: 844.8 seconds

  📋 Fu